# Systematic Feature Selection: Season-Specific Models for Cattle Mortality

## Objective

Compare **season-specific models** vs. **pooled models with season interactions** to determine:
1. Do different heat stress metrics matter in different seasons?
2. Should we build separate models for Summer vs. Winter vs. Spring/Fall?
3. Which seasons are most predictable from heat metrics?

## Scientific Rationale

**Season-specific modeling** is motivated by:
- **Physiological differences**: Cattle are heat-acclimated in summer, vulnerable to sudden heat in spring
- **Threshold differences**: 30°C may be critical in spring but moderate in summer
- **Management differences**: Summer = active cooling; Winter = minimal intervention
- **Mortality patterns**: Heat-related deaths peak June-August

## Hypotheses

**H1**: Summer model will have highest R² (strongest heat signal)

**H2**: Summer model will prioritize VPD and nighttime recovery (evaporative cooling critical)

**H3**: Spring model will prioritize extreme heat events (lack of acclimation)

**H4**: Season-specific models will outperform pooled model (season interactions insufficient)

**H5**: Winter model will have lowest R² (heat stress less relevant)

---

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, ElasticNetCV
from sklearn.feature_selection import RFE, RFECV
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Statistical modeling (optional - for advanced diagnostics)
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("⚠ statsmodels not available - some advanced features will be skipped")

# Configuration
import sys
sys.path.append('../../')
from config import (
    PROCESSED_WEEKLY_DIR, MASK_FILE, CATTLE_DATA_FILE,
    CATTLE_REGION_IDS, SEASONS
)

# Plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
np.random.seed(42)

# Output directory
OUTPUT_DIR = Path('../../figures/feature_selection_seasonal')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries loaded successfully")
print(f"\\nSeason definitions:")
for season, months in SEASONS.items():
    print(f"  {season}: {months}")

Libraries loaded successfully
\nSeason definitions:
  Winter: [12, 1, 2]
  Spring: [3, 4, 5]
  Summer: [6, 7, 8]
  Fall: [9, 10, 11]


## 1. Load Data and Create Season-Stratified Datasets

In [2]:
# Load weekly climate data
print("Loading weekly climate data...")
ds_night = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')
ds_day = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')
ds_vpd = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_vpd.nc')

week_dates = ds_night['week'].values

# Load state mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask.load()
ds_mask.close()

# Helper function
def compute_regional_mean(data, state_ids, state_mask):
    combined_mask = xr.zeros_like(state_mask, dtype=bool)
    for state_id in state_ids:
        combined_mask = combined_mask | (state_mask == state_id)
    masked_data = data.where(combined_mask)
    return masked_data.mean(dim=['lat', 'lon']).astype(np.float64)

region_4_ids = CATTLE_REGION_IDS['region_4']
region_6_ids = CATTLE_REGION_IDS['region_6']

# Compute regional metrics
climate_dfs = []
for region_num, state_ids in [(4, region_4_ids), (6, region_6_ids)]:
    df = pd.DataFrame({
        'week_ending': pd.to_datetime(week_dates),
        'region': region_num,
        'mean_daytime_hours_above_30': compute_regional_mean(ds_day['hours_above_30'], state_ids, state_mask).values,
        'mean_daytime_hours_above_35': compute_regional_mean(ds_day['hours_above_35'], state_ids, state_mask).values,
        'mean_nighttime_hours_above_21': compute_regional_mean(ds_night['hours_above_21'], state_ids, state_mask).values,
        'mean_nighttime_hours_above_24': compute_regional_mean(ds_night['hours_above_24'], state_ids, state_mask).values,
        'mean_vpd_mean': compute_regional_mean(ds_vpd['vpd_mean'], state_ids, state_mask).values,
        'mean_vpd_max': compute_regional_mean(ds_vpd['vpd_max'], state_ids, state_mask).values,
    })
    climate_dfs.append(df)

climate_data = pd.concat(climate_dfs, ignore_index=True)

# Load cattle data
cattle_data = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])
cattle_data = cattle_data.rename(columns={'date': 'week_ending'})

# Reshape cattle data
cattle_r4 = cattle_data[['week_ending']].copy()
cattle_r4['region'] = 4
cattle_r4['total_beef_dairy'] = cattle_data['region_4_beef_dairy']

cattle_r6 = cattle_data[['week_ending']].copy()
cattle_r6['region'] = 6
cattle_r6['total_beef_dairy'] = cattle_data['region_6_beef_dairy']

cattle_regional = pd.concat([cattle_r4, cattle_r6], ignore_index=True)

# Merge
cattle_heat = pd.merge(cattle_regional, climate_data, on=['week_ending', 'region'], how='inner')
cattle_heat = cattle_heat.dropna()

# Add temporal features
cattle_heat['year'] = cattle_heat['week_ending'].dt.year
cattle_heat['month'] = cattle_heat['week_ending'].dt.month

def get_season(month):
    if month in SEASONS['Winter']:
        return 'Winter'
    elif month in SEASONS['Spring']:
        return 'Spring'
    elif month in SEASONS['Summer']:
        return 'Summer'
    else:
        return 'Fall'

cattle_heat['season'] = cattle_heat['month'].apply(get_season)

cattle_heat = cattle_heat.sort_values(['region', 'week_ending']).reset_index(drop=True)

print(f"\nMerged data: {len(cattle_heat)} region-weeks")
print(f"\nSamples per season:")
print(cattle_heat['season'].value_counts().sort_index())

Loading weekly climate data...

Merged data: 4382 region-weeks

Samples per season:
season
Fall      1092
Spring    1104
Summer    1106
Winter    1080
Name: count, dtype: int64


## 2. Feature Engineering with Season-Specific Focus

In [3]:
# Base features
heat_metrics = [
    'mean_daytime_hours_above_30',
    'mean_daytime_hours_above_35',
    'mean_nighttime_hours_above_21',
    'mean_nighttime_hours_above_24',
    'mean_vpd_mean',
    'mean_vpd_max'
]

# Create lagged features (1-2 weeks for seasonal models)
for lag in [1, 2]:
    for metric in heat_metrics:
        cattle_heat[f'{metric}_lag{lag}'] = cattle_heat.groupby('region')[metric].shift(lag)

print(f"Created {2 * len(heat_metrics)} lagged features")

# Interaction: heat × VPD
cattle_heat['heat_vpd_interaction'] = (
    cattle_heat['mean_daytime_hours_above_30'] * cattle_heat['mean_vpd_mean']
)

# Regional indicator
cattle_heat['region_indicator'] = cattle_heat['region']

# Define feature list
lagged_features = [col for col in cattle_heat.columns if 'lag' in col]
interaction_features = ['heat_vpd_interaction']
regional_features = ['region_indicator']

all_features = heat_metrics + lagged_features + interaction_features + regional_features
target_var = 'total_beef_dairy'

print(f"\nTotal features: {len(all_features)}")
print(f"  Base: {len(heat_metrics)}")
print(f"  Lagged: {len(lagged_features)}")
print(f"  Interactions: {len(interaction_features)}")
print(f"  Regional: {len(regional_features)}")

Created 12 lagged features

Total features: 20
  Base: 6
  Lagged: 12
  Interactions: 1
  Regional: 1


## 3. Create Season-Stratified Train/Test Splits

In [4]:
# Prepare base data
model_data = cattle_heat[all_features + [target_var, 'year', 'season']].dropna().copy()

# Create datasets for each season
season_datasets = {}

for season in ['Winter', 'Spring', 'Summer', 'Fall']:
    season_data = model_data[model_data['season'] == season].copy()
    
    # Train/test split
    train_mask = season_data['year'] <= 2015
    test_mask = season_data['year'] > 2015
    
    X_train = season_data.loc[train_mask, all_features]
    y_train = season_data.loc[train_mask, target_var]
    X_test = season_data.loc[test_mask, all_features]
    y_test = season_data.loc[test_mask, target_var]
    
    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    season_datasets[season] = {
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
        'y_test': y_test,
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled,
        'scaler': scaler
    }
    
    print(f"\n{season}:")
    print(f"  Train: {len(X_train)} samples")
    print(f"  Test: {len(X_test)} samples")

print("\nSeason-stratified datasets prepared")


Winter:
  Train: 818 samples
  Test: 258 samples

Spring:
  Train: 842 samples
  Test: 262 samples

Summer:
  Train: 842 samples
  Test: 264 samples

Fall:
  Train: 832 samples
  Test: 260 samples

Season-stratified datasets prepared


## 4. Pooled Model with Season Indicators

In [5]:
# Create pooled dataset with season dummies
pooled_data = model_data.copy()
pooled_data = pd.get_dummies(pooled_data, columns=['season'], prefix='season', drop_first=False)

# Features for pooled model
season_dummies = [col for col in pooled_data.columns if col.startswith('season_')]
pooled_features = all_features + season_dummies

# Train/test split
train_mask = pooled_data['year'] <= 2015
test_mask = pooled_data['year'] > 2015

X_train_pooled = pooled_data.loc[train_mask, pooled_features]
y_train_pooled = pooled_data.loc[train_mask, target_var]
X_test_pooled = pooled_data.loc[test_mask, pooled_features]
y_test_pooled = pooled_data.loc[test_mask, target_var]

# Standardize
scaler_pooled = StandardScaler()
X_train_pooled_scaled = scaler_pooled.fit_transform(X_train_pooled)
X_test_pooled_scaled = scaler_pooled.transform(X_test_pooled)

print(f"Pooled model:")
print(f"  Train: {len(X_train_pooled)} samples")
print(f"  Test: {len(X_test_pooled)} samples")
print(f"  Features: {len(pooled_features)} (including {len(season_dummies)} season indicators)")

Pooled model:
  Train: 3334 samples
  Test: 1044 samples
  Features: 24 (including 4 season indicators)


## 5. Model Comparison Framework

For each season, we'll test:
1. Current week only (no lags)
2. With 1-week lag
3. With all lags (1-2 weeks)
4. Forward selection (BIC)
5. Backward elimination (BIC)
6. LASSO-CV
7. Elastic Net
8. Best subset (if feasible)
9. Ridge (baseline regularized)
10. RFE
11. Stepwise (AIC)
12. Theory-driven (VPD + extreme heat + recovery)

Plus 4 pooled models for comparison.

In [6]:
# Helper function
def calculate_metrics(y_true, y_pred, n_features, n_samples):
    """Calculate comprehensive model metrics"""
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    n = n_samples
    k = n_features + 1
    rss = np.sum((y_true - y_pred) ** 2)
    
    aic = n * np.log(rss / n) + 2 * k
    bic = n * np.log(rss / n) + k * np.log(n)
    
    return {
        'R2': r2,
        'Adj_R2': adj_r2,
        'RMSE': rmse,
        'MAE': mae,
        'AIC': aic,
        'BIC': bic,
        'N_Features': n_features
    }

# Initialize results storage
all_seasonal_results = []

print("Helper functions defined")
print("\nReady to run models for each season...")

Helper functions defined

Ready to run models for each season...
